## Notebook: LLM Mechanics and Prompt Engineering

In [1]:
# Imagine Walmart wants to use Generative AI to automate everyday business activities, such as understanding customer complaints, analysing product feedback, making inventory decisions, reviewing supplier communications, and checking product listings.

# Normally, employees would need to read information, understand the situation, make decisions, and prepare responses manually.

# The challenge is: How can Walmart use Large Language Models (LLMs) to perform these activities accurately, consistently, and cost-effectively?

# Simply connecting an LLM to a business application is not enough.

# For example:
# - What happens if an AI model misunderstands a customer complaint?
# - What if it forgets an inventory threshold mentioned earlier?
# - What if it gives different answers every time?
# - What if it produces a paragraph when the application expects JSON?
# - What if unnecessary tokens increase API costs?

# This notebook teaches us how to address these challenges by understanding how LLMs work and how to communicate effectively with them using prompt engineering.

In [2]:
# What are we trying to build or achieve?

# Part 1 — Understand how an LLM works
# Learn tokenization, context windows, and temperature so we understand API cost, conversational memory, and output variability.

# Part 2 — Learn how to write effective prompts
# Practise zero-shot, few-shot, chain-of-thought, role prompting, and structured output using business scenarios.

# Part 3 — Apply everything in a hands-on exercise
# Build an AI-based product listing classifier that identifies product categories, checks possible policy violations, and returns structured results.

In [3]:
# ## Setup — Load Libraries and API Keys

import os
import json
from openai import OpenAI
from dotenv import load_dotenv
import tiktoken

load_dotenv(override=True)

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("OK Libraries loaded. OpenAI client ready.")
print(f"   Using model: gpt-4o-mini (cost-efficient for labs)")
print(f"   Using model: gpt-4o (for advanced demonstrations)")


OK Libraries loaded. OpenAI client ready.
   Using model: gpt-4o-mini (cost-efficient for labs)
   Using model: gpt-4o (for advanced demonstrations)


---
# Part 1: LLM Mechanics

Before we build production systems with LLMs, we need to understand exactly how they work — not at a research level, but at the level that matters for engineering decisions.

Three mechanics that every AI engineer must understand:
1. **Tokenization** — What the model actually reads
2. **Context Window** — What the model can remember
3. **Temperature** — How the model chooses its next word


## 1.1 Tokenization

An LLM does not read words. It reads **tokens**.

A token is roughly 3–4 characters in English. Common words are one token. Rare words, proper nouns, and code identifiers are often split into multiple tokens.

**Why this matters for engineering:**
- API costs are charged per token, not per word
- Context window limits are measured in tokens
- Poorly structured prompts waste tokens and increase cost

Let's see exactly how Walmart-relevant text gets tokenized.


In [4]:
# Tokenization Demo
enc = tiktoken.encoding_for_model("gpt-4o")

texts = [
    "Walmart",
    "Walmart processes 2.3 million customer transactions per hour.",
    "Supply chain optimization using LangGraph and LLM-as-judge evaluation.",
    "SKU-7829341-B",   # A product identifier — watch how this is split
    "def run_agent(state: AgentState) -> AgentState:",  # Code
]

print(f"Tokenizer: {enc.name}")
print(f"Vocabulary size: {enc.n_vocab:,} tokens")
print()
print(f"{'Text':<55} | {'Tokens':>6} | {'Token IDs (first 8)'}")
print("-" * 90)

for text in texts:
    token_ids = enc.encode(text)
    token_words = [enc.decode([t]) for t in token_ids]
    preview_ids = str(token_ids[:8]) + ("..." if len(token_ids) > 8 else "")
    print(f"{text[:54]:<55} | {len(token_ids):>6} | {preview_ids}")

print()
print("Key insight: 'SKU-7829341-B' is split into multiple tokens.")
print("Structured identifiers cost more tokens than natural text.")


Tokenizer: o200k_base
Vocabulary size: 200,019 tokens

Text                                                    | Tokens | Token IDs (first 8)
------------------------------------------------------------------------------------------
Walmart                                                 |      2 | [54, 40173]
Walmart processes 2.3 million customer transactions pe  |     13 | [54, 40173, 14340, 220, 17, 13, 18, 5749]...
Supply chain optimization using LangGraph and LLM-as-j  |     14 | [49473, 13464, 34658, 2360, 27830, 9922, 326, 451]...
SKU-7829341-B                                           |      6 | [72089, 12, 47574, 47717, 16, 8287]
def run_agent(state: AgentState) -> AgentState:         |     12 | [1314, 2461, 62035, 16114, 25, 28237, 1881, 8]...

Key insight: 'SKU-7829341-B' is split into multiple tokens.
Structured identifiers cost more tokens than natural text.


### Token Cost Calculator

At $0.150 per 1M input tokens (gpt-4o-mini), even small inefficiencies in prompts add up at Walmart's scale.


In [5]:
# Token Cost Calculator
def estimate_cost(text: str, model: str = "gpt-4o-mini", direction: str = "input") -> dict:
    """Estimate the token count and API cost for a given text."""
    enc = tiktoken.encoding_for_model("gpt-4o")  # gpt-4o enc works for gpt-4o-mini too
    token_count = len(enc.encode(text))
    
    # Pricing as of 2026 (approximate, verify at platform.openai.com)
    pricing = {
        "gpt-4o-mini": {"input": 0.150, "output": 0.600},   # per 1M tokens
        "gpt-4o":      {"input": 2.50,  "output": 10.00},
    }
    
    price_per_million = pricing.get(model, {}).get(direction, 0)
    cost_usd = (token_count / 1_000_000) * price_per_million
    
    return {
        "text_chars": len(text),
        "token_count": token_count,
        "chars_per_token": round(len(text) / token_count, 2) if token_count > 0 else 0,
        "cost_usd": round(cost_usd, 8),
        "cost_per_1000_calls": round(cost_usd * 1000, 4),
    }

# Compare prompt efficiency
prompts = {
    "Verbose system prompt": """You are an AI assistant working at Walmart. Your job is to help customer service 
    representatives answer questions. You should always be helpful, professional, and accurate. You must never 
    make up information that you do not know. Always be polite and courteous to customers.""",
    
    "Concise system prompt": """You are a Walmart customer service AI. Be accurate, professional, brief.""",
}

print("Prompt Efficiency Comparison (gpt-4o-mini, input tokens):")
print("-" * 70)
for name, prompt in prompts.items():
    stats = estimate_cost(prompt, "gpt-4o-mini", "input")
    print(f"\n{name}:")
    print(f"  Characters : {stats['text_chars']}")
    print(f"  Tokens     : {stats['token_count']}")
    print(f"  Cost/call  : ${stats['cost_usd']:.6f}")
    print(f"  Cost/1000 calls: ${stats['cost_per_1000_calls']}")

print()
print("At 1M calls/day (Walmart scale), the difference compounds significantly.")


Prompt Efficiency Comparison (gpt-4o-mini, input tokens):
----------------------------------------------------------------------

Verbose system prompt:
  Characters : 284
  Tokens     : 55
  Cost/call  : $0.000008
  Cost/1000 calls: $0.0083

Concise system prompt:
  Characters : 72
  Tokens     : 15
  Cost/call  : $0.000002
  Cost/1000 calls: $0.0023

At 1M calls/day (Walmart scale), the difference compounds significantly.


## 1.2 Context Window

The context window is the model's working memory — everything it can "see" at once.

| Model | Context Window |
|---|---|
| gpt-4o-mini | 128,000 tokens |
| gpt-4o | 128,000 tokens |
| Claude Sonnet | 200,000 tokens |

**Why this matters:**
- Long documents must be chunked if they exceed the window
- Every message in a chat adds to the context and costs tokens
- Agents with long tool call histories can hit context limits mid-task

Let's demonstrate how context window enables multi-turn memory.


In [7]:
# Context Window Demo — Multi-turn Memory
print("=== Demo: How Context Window Creates Memory ===\n")

# Conversation 1: Without context history (model has no memory)
print("NO Without context (no message history passed):")
response_no_context = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "What inventory threshold did I mention?"}
    ],
    max_tokens=80,
    temperature=0
)
print(f"   Model says: {response_no_context.choices[0].message.content.strip()}\n")

# Conversation 2: With full context history passed
print("OK With context (full message history passed):")
conversation_history = [
    {"role": "user",      "content": "Our reorder threshold for SKU-7829 is 500 units."},
    {"role": "assistant", "content": "Understood. The reorder threshold for SKU-7829 is 500 units."},
    {"role": "user",      "content": "And for SKU-9102 it's 250 units."},
    {"role": "assistant", "content": "Got it. SKU-9102 reorder threshold is 250 units."},
    {"role": "user",      "content": "What inventory threshold did I mention for SKU-7829?"},
]

response_with_context = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=conversation_history,
    max_tokens=80,
    temperature=0
)
print(f"   Model says: {response_with_context.choices[0].message.content.strip()}\n")

# Show context size
enc = tiktoken.encoding_for_model("gpt-4o")
total_tokens = sum(len(enc.encode(m["content"])) for m in conversation_history)
print(f"STATS Context used: ~{total_tokens} tokens")
print(f"   Window capacity: 128,000 tokens")
print(f"   Remaining: {128000 - total_tokens:,} tokens")


=== Demo: How Context Window Creates Memory ===

NO Without context (no message history passed):
   Model says: I'm sorry, but I don't have access to previous conversations or specific details about your inventory threshold. If you provide me with more context or details, I would be happy to help you with your inquiry!

OK With context (full message history passed):
   Model says: You mentioned that the inventory reorder threshold for SKU-7829 is 500 units.

STATS Context used: ~66 tokens
   Window capacity: 128,000 tokens
   Remaining: 127,934 tokens


## 1.3 Temperature

Temperature controls **randomness** in the model's output.

| Temperature | Behaviour | Use Case |
|---|---|---|
| 0.0 | Deterministic, always most likely token | Classification, extraction, code generation |
| 0.2–0.5 | Slightly varied, mostly consistent | Summaries, factual Q&A, structured outputs |
| 0.7–1.0 | Creative, more varied | Marketing copy, ideation, creative writing |
| > 1.2 | Highly random, may be incoherent | Rarely useful in production |

**Rule of thumb for production systems:** Use the lowest temperature that produces acceptable output quality.


In [8]:
# Temperature Demo
import time

prompt = "In one sentence, suggest how Walmart can use AI to improve store checkout speed."

print("=== Temperature Effect on Output Variability ===\n")
print(f"Prompt: \"{prompt}\"\n")

for temp in [0.0, 0.5, 1.2]:
    print(f"Temperature = {temp}:")
    responses = []
    for i in range(3):
        r = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=temp,
            max_tokens=60,
        )
        responses.append(r.choices[0].message.content.strip())
        time.sleep(0.3)  # Avoid rate limiting
    
    for i, resp in enumerate(responses, 1):
        print(f"  Run {i}: {resp}")
    print()

print("Observation: At temp=0.0, all 3 runs give identical output.")
print("At temp=1.2, outputs diverge significantly — unpredictable for production.")


=== Temperature Effect on Output Variability ===

Prompt: "In one sentence, suggest how Walmart can use AI to improve store checkout speed."

Temperature = 0.0:
  Run 1: Walmart can implement AI-powered checkout systems that utilize computer vision and machine learning to automatically scan and process items in customers' carts, significantly reducing wait times at checkout.
  Run 2: Walmart can implement AI-powered checkout systems that utilize computer vision and machine learning to automatically scan and process items in customers' carts, significantly reducing wait times at checkout.
  Run 3: Walmart can implement AI-powered checkout systems that utilize computer vision and machine learning to automatically scan and process items in customers' carts, significantly reducing wait times at checkout.

Temperature = 0.5:
  Run 1: Walmart can implement AI-powered automated checkout systems that utilize computer vision and machine learning to streamline the scanning process, reducing wait

---
# Part 2: Prompt Engineering

Prompt engineering is the practice of designing inputs to LLMs to consistently produce the outputs you need.

It is not a workaround. It is a core engineering discipline. A well-engineered prompt can often outperform a fine-tuned model — at a fraction of the cost.

We will cover 5 patterns, each with a Walmart business use case.


In [11]:
# I go to a restaurant which has a menu card on the table, and I call the waiter and ask him:
#     BRING ME FOOD.

# What do you think the waiter will get for me?

## 2.1 Zero-Shot Prompting

Ask the model to perform a task with no examples. Works well when:
- The task is common in the model's training data
- The output format is simple

**Walmart use case:** Classify the urgency of a customer support ticket.


In [12]:
# Zero-Shot Prompting
def zero_shot_classify(complaint: str) -> str: # complaint="My item arrived damaged and I need a replacement immediately."
    """Classify complaint severity with no examples — pure zero-shot."""
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a Walmart customer support triage system. "
                    "Classify each complaint as CRITICAL, HIGH, MEDIUM, or LOW severity. "
                    "Reply with exactly one word: the severity level."
                )
            },
            {"role": "user", "content": f"Complaint: {complaint}"}
        ],
        temperature=0,
        max_tokens=10,
    )
    return response.choices[0].message.content.strip()

complaints = [
    "My item arrived damaged and I need a replacement immediately.",
    "Website was slow this morning but seems fine now.",
    "I was charged twice for the same order — my bank is disputing it.",
    "The packaging could be more eco-friendly.",
    "I can't log into my Walmart+ account and my groceries won't deliver.",
]

print("Zero-Shot Complaint Classification:")
print("-" * 65)
for c in complaints: # c="My item arrived damaged and I need a replacement immediately."
    severity = zero_shot_classify(c) # zero_shot_classify("My item arrived damaged and I need a replacement immediately.")
    emoji = {"CRITICAL": "🔴", "HIGH": "🟠", "MEDIUM": "🟡", "LOW": "🟢"}.get(severity, "⚪")
    print(f"{emoji} {severity:<10} | {c[:55]}")


Zero-Shot Complaint Classification:
-----------------------------------------------------------------
🟠 HIGH       | My item arrived damaged and I need a replacement immedi
🟢 LOW        | Website was slow this morning but seems fine now.
🟠 HIGH       | I was charged twice for the same order — my bank is dis
🟢 LOW        | The packaging could be more eco-friendly.
🟠 HIGH       | I can't log into my Walmart+ account and my groceries w


## 2.2 Few-Shot Prompting

Provide examples of input → output pairs before the actual task. The model learns the pattern from your examples.

**When to use over zero-shot:**
- You need a specific output format
- The task has domain-specific nuances the model might not know
- Zero-shot gives inconsistent results

**Walmart use case:** Extract structured data from unstructured customer feedback.


In [13]:
# Few-Shot Prompting
def few_shot_extract(feedback: str) -> dict: # feedback="The Equate pain relief tablets work great — I use them every day and they're half the price of brand name."
    """Extract structured data from feedback using few-shot examples."""
    
    few_shot_prompt = """Extract product information from customer feedback.
Return a JSON object with keys: product_name, sentiment (positive/negative/neutral), issue (or null), rating_implied (1-5).
Return ONLY valid JSON. Do not include markdown or extra text.

Example 1:
Feedback: "The Great Value coffee maker stopped working after 3 weeks. Very disappointing."
Output: {{"product_name": "Great Value coffee maker", "sentiment": "negative", "issue": "stopped working after 3 weeks", "rating_implied": 1}}

Example 2:
Feedback: "Absolutely love my new Onn. TV! Picture quality is stunning and setup was easy."
Output: {{"product_name": "Onn. TV", "sentiment": "positive", "issue": null, "rating_implied": 5}}

Example 3:
Feedback: "The Sam's Choice pasta is decent. Nothing special but good value for the price."
Output: {{"product_name": "Sam's Choice pasta", "sentiment": "neutral", "issue": null, "rating_implied": 3}}

Now extract from:
Feedback: "{feedback}"
Output:"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": few_shot_prompt.format(feedback=feedback)}],
        response_format={"type": "json_object"},
        temperature=0,
        max_tokens=120,
    )
    
    raw = response.choices[0].message.content.strip()
    return json.loads(raw)

feedbacks = [
    "The Equate pain relief tablets work great — I use them every day and they're half the price of brand name.",
    "My Walmart+ order was 2 hours late and the frozen food was partially thawed. Very frustrated.",
    "George men's jeans fit perfectly and the material feels durable. Will buy again.",
]

print("Few-Shot Structured Extraction:")
print("=" * 65)
for fb in feedbacks: # fb= "The Equate pain relief tablets work great — I use them every day and they're half the price of brand name."
    result = few_shot_extract(fb) # few_shot_extract("The Equate pain relief tablets work great — I use them every day and they're half the price of brand name.")
    print(f"\nFeedback: {fb[:60]}...")
    print(f"Extracted: {json.dumps(result, indent=2)}")

Few-Shot Structured Extraction:

Feedback: The Equate pain relief tablets work great — I use them every...
Extracted: {
  "product_name": "Equate pain relief tablets",
  "sentiment": "positive",
  "issue": null,
  "rating_implied": 5
}

Feedback: My Walmart+ order was 2 hours late and the frozen food was p...
Extracted: {
  "product_name": "Walmart+",
  "sentiment": "negative",
  "issue": "order was 2 hours late and the frozen food was partially thawed",
  "rating_implied": 1
}

Feedback: George men's jeans fit perfectly and the material feels dura...
Extracted: {
  "product_name": "George men's jeans",
  "sentiment": "positive",
  "issue": null,
  "rating_implied": 5
}


## 2.3 Chain-of-Thought (CoT) Prompting

Tell the model to **think step by step** before giving a final answer. This forces the model to reason through multi-step problems rather than pattern-match to the nearest answer.

**When to use:**
- Multi-step reasoning (math, logic, planning)
- Decisions that require considering multiple factors
- Any situation where you'd want a human to "show their work"

**Walmart use case:** Supply chain reorder decision with multiple constraints.


In [14]:
# Without Chain-of-Thought Prompting
# Prompt: What is the area of a rectangle with length 10 and width 5?
# Answer: 50

# With Chain-of-Thought Prompting
# Prompt: What is the area of a rectangle with length 10 and width 5? Think step by step.
# Answer:
# We need to calculate the area of a rectangle.
# Given the length of rectangle is 10
# and the width of rectangle is 5
# The formula for area of rectangle is length * width
# Substituting the values, we get area = 10 * 5 = 50
# Therefore, the area of the rectangle is 50.

In [15]:
# Chain-of-Thought Prompting

# ── WITHOUT Chain-of-Thought ──
no_cot_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": """
A Walmart distribution center has the following situation:
- Current stock of Product X: 800 units
- Average daily demand: 65 units/day
- Lead time to restock: 8 days
- Safety stock required: 4 days of demand
- Upcoming promotional event in 5 days expected to triple demand for 3 days

Should we place a reorder today? Answer yes or no.
"""}
    ],
    temperature=0,
    max_tokens=20,
)
print("WITHOUT Chain-of-Thought:")
print(f"  Answer: {no_cot_response.choices[0].message.content.strip()}\n")

# ── WITH Chain-of-Thought ──
cot_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": """
A Walmart distribution center has the following situation:
- Current stock of Product X: 800 units
- Average daily demand: 65 units/day
- Lead time to restock: 8 days
- Safety stock required: 4 days of demand
- Upcoming promotional event in 5 days expected to triple demand for 3 days

Should we place a reorder today? Think step by step, calculate each quantity explicitly, then give a final recommendation with justification.
"""}
    ],
    temperature=0,
    max_tokens=400,
)
print("WITH Chain-of-Thought:")
print(cot_response.choices[0].message.content.strip())


WITHOUT Chain-of-Thought:
  Answer: To determine whether to place a reorder today, we need to calculate the stock situation considering the upcoming promotional

WITH Chain-of-Thought:
To determine whether to place a reorder today for Product X at the Walmart distribution center, we need to analyze the current stock, demand, lead time, safety stock, and the impact of the upcoming promotional event. Let's break this down step by step.

### Step 1: Calculate the Safety Stock
Safety stock is calculated based on the average daily demand and the number of days of safety stock required.

- **Average daily demand**: 65 units/day
- **Safety stock required**: 4 days of demand

\[
\text{Safety Stock} = \text{Average Daily Demand} \times \text{Safety Stock Days} = 65 \, \text{units/day} \times 4 \, \text{days} = 260 \, \text{units}
\]

### Step 2: Calculate the Total Stock Requirement
The total stock requirement includes the safety stock and the stock needed to cover demand during the lead time.


## 2.4 Role Prompting

Define a specific persona and expertise for the model. This activates relevant knowledge and adjusts the communication style to match the role.

**When to use:**
- You need domain-specific language and framing
- The audience for the output is specific (executives vs engineers vs customers)
- Tone and perspective matter

**Walmart use case:** Get different analyses of the same problem from different roles.


In [16]:
# Role Prompting — Same Question, Different Roles

business_problem = """
Walmart is considering deploying an AI agent to automate supplier negotiation emails. 
The agent would draft, send, and respond to routine supplier communications autonomously.
What are the key considerations?
"""

roles = {
    "AI Architect": "You are a Senior AI Systems Architect at Walmart Global Tech with 15 years of experience in enterprise AI deployments. Focus on technical architecture, reliability, and system design.",
    "Risk Officer": "You are the Chief Risk Officer at a Fortune 500 retailer. Focus on legal liability, compliance, audit trails, and reputational risk.",
    "Finance Lead": "You are VP of Finance at Walmart. Focus on ROI, cost savings, implementation costs, and financial governance.",
}

for role_name, system_prompt in roles.items():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": business_problem},
        ],
        temperature=0.2,
        max_tokens=200,
    )
    print(f"\n{'=' * 65}")
    print(f"Role: {role_name}")
    print('=' * 65)
    print(response.choices[0].message.content.strip())



Role: AI Architect
Deploying an AI agent to automate supplier negotiation emails involves several key considerations across various dimensions, including technical architecture, reliability, compliance, and user experience. Here are the critical aspects to consider:

### 1. **Technical Architecture**
   - **Natural Language Processing (NLP)**: The AI agent should leverage advanced NLP models to understand and generate human-like text. Consider using transformer-based models (e.g., GPT, BERT) fine-tuned for negotiation contexts.
   - **Integration with Existing Systems**: Ensure seamless integration with existing email systems (e.g., Outlook, Gmail) and ERP systems (e.g., SAP, Oracle) to access supplier data and historical communications.
   - **Data Storage and Management**: Design a robust database to store communication history, supplier profiles, and negotiation outcomes. Ensure it supports efficient querying and retrieval.
   - **Scalability**: The architecture should be scalable 

## 2.5 Structured Output Prompting

Force the model to return machine-readable JSON that conforms to a specific schema. In production, this eliminates fragile string parsing.

**The modern approach:** Use OpenAI's native structured output with Pydantic models — guaranteed schema compliance.

**Walmart use case:** Extract a shipment incident report from a free-text description.
